In [ ]:
# pylint: disable=wrong-import-position
# pylint: disable=wrong-import-order
# pylint: disable=invalid-name
# pylint: disable=redefined-outer-name
"""Demo decision process built with polycog"""

# Decision Process


Polycog agents use a **decision process** as their core design pattern to organize and orchestrate task execution. A decision process is a declarative, state-driven execution machine that continually matches a state against a collection of independent operators to determine the next step. Instead of executing code along a fixed, linear path, a decision process runs an iterative loop on every system tick. From a developer's perspective, configuring a decision process is as straightforward as programming a traditional workflow. However, the underlying execution model is fundamentally different:

> **Imperative vs. Declarative:** Instead of hardcoding an explicit, rigid sequence of steps in a workflow, you define **decision criteria** declaratively. Polycog's runtime orchestration engine (`cognition`) continuously evaluates these criteria against a live state to dynamically compute, execute, and adjust the workflow in real-time. ([details](#declarative-vs-imperative-control-of-decisions))

Decision processes are similar in spirit to a combination of [event-driven](https://en.wikipedia.org/wiki/Event-driven_programming) and [data-driven](https://en.wikipedia.org/wiki/Data-driven_programming) programming paradigms but are designed for agentic systems. They continually evaluate agent state to determine the next steps of reasoning or action, each of which mutates the state, leading to the next decision. 

## Running Example: Home Assistant Agent
To illustrate how a decision process operates, we will build a smart home agent using the `cognition` library. While a production-grade agent might manage an entire ecosystem of connected appliances (smart plugs, dishwashers, washing machines), our example focuses on a simplified environment: a single room with a single lamp. This lamp features a manual switch that can be toggled by either the homeowner or the agent.

Our goal is to implement a decision process for the agent's **MIDNIGHT** mode: when the agent is in **MIDNIGHT** mode, the agent should automatically flip the switch off if the lamp is currently on.

In [ ]:
## All necessary imports
from enum import Enum, auto

from cognition import DecisionProcess, IOContainer, Operator

## Core concepts

`cognition` relies on three core concepts: state, operator, and a termination check.

### 1. State
State captures the current truth of the environment and tracks the agent's progress through its task execution. 

You can define and customize the state using standard Python classes. Use fields and methods that best reflect the environment your agent is working in.

In [ ]:
from dataclasses import dataclass


class Mode(Enum):
    """enum to maintain the mode home assistant is operating in"""

    DAY = auto()
    EVENING = auto()
    MIDNIGHT = auto()


class Lamp(Enum):
    """enum to capture lamp state"""

    ON = auto()
    OFF = auto()


class Switch(Enum):
    """enum to capture switch state"""

    ENABLED = auto()
    DISABLED = auto()


@dataclass
class HomeState:
    """describe the current state"""

    mode: Mode
    switch: Switch
    actuated: bool = False

    @property
    def lamp(self) -> Lamp:
        """automatically set the lamp state"""
        if self.switch is Switch.ENABLED:
            return Lamp.ON
        return Lamp.OFF

### 2. Operator

`operator` is a core base class provided by the `cognition` library. You inherit from this class to define your agent's behavior. Your derived classes describe how to transform the state. 

#### The anatomy of an operator

When defining a custom `Operator`, you must override two methods that dictate the conditional execution flow. Both methods accept the same two arguments that are supplied by the `cognition` engine. 
- `state: HomeState`: Your custom-defined object capturing the agent's current internal working memory.
- `io: IOContainer`: An IOContainer connected to the external sources and channels of data including peripherals (sensors and actuators) and other decision processes. We will learn about how to use these in later tutorials. 

`Operator` has two lifecycle methods that implement how it should be considered by the `cognition` engine. 

* **`can_perform` ("when"):** Evaluates the current state (and environmental inputs). It returns a bool indicating if this operator is eligible to execute. If the bool is true, the engine *proposes* the operator. 
* **`perform` ("then"):** Defines the outcome. It *applies* the operator by mutating the state when this operator is selected by the engine.

In [ ]:
class TurnLightOff(Operator[HomeState]):
    """
    when: it is MIDNIGHT mode and the lamp is ON.
    then: disable the switch (and turn-off the lamp), and set the actuated flag.
    """

    def can_perform(self, state: HomeState, _io: IOContainer) -> bool:
        """propose when it is MIDNIGHT mode and the lamp is ON"""
        return state.mode is Mode.MIDNIGHT and state.lamp is Lamp.ON

    def perform(self, state: HomeState, _io: IOContainer) -> None:
        """apply to disable the switch (and turn-off the lamp), and set the actuated flag."""
        state.switch = Switch.DISABLED
        state.actuated = True

---

> 📌 **Design Note:** Notice how `can_perform` only *reads* data to make a boolean decision, while `perform` is where the actual state *mutation* happens. Keeping these phases purely decoupled allows the `cognition` library's underlying conflict-resolution engine to evaluate multiple competing operators before committing to an action. 

### 3. Defining the termination conditions

The final step in implementing a decision process is defining a **terminal check**. 

One way to implement it is designing a function that inspects the state to determine whether the active execution cycle should conclude. When this method returns `True`, it signals to the `cognition` engine that a specific goal or a desired outcome has been met. The engine then halts the current decision-making iteration and passes control back.

You will encounter other ways to specifying termination checks as you progress through the tutorials. 


In [ ]:
def is_terminal(state: HomeState, _io: IOContainer) -> bool:
    """
    Evaluates if the current decision cycle has reached a terminal step.

    Returns True if an operator successfully flagged 'actuated', indicating
    a state mutation or external action command has been processed.
    """
    actuated = state.actuated
    state.actuated = False
    return actuated

---

💡 Termination: By checking `state.actuated` is True, the engine ensures that as soon as a meaningful action is determined and staged by an operator (like `TurnLightOff`), the decision process yields immediately so the agent can interact with the environment.

## Assembling the decision process 

With the `State`, `Operator` rules, and `is_terminal` method implemented, you can now put them together into `DecisionProcess`. 



In [ ]:
# 1. Initialize state in MIDNIGHT mode with lamp turn on and switch enabled.
state: HomeState = HomeState(mode=Mode.MIDNIGHT, switch=Switch.ENABLED)

# 2. Instantiate the decision process, passing it a function that reinitializes HomeState both now AND whenever the decision process is reinitialized.
dp: DecisionProcess[HomeState] = DecisionProcess(lambda: state)

# 3. Register your operator and give it a readable name
dp.add_operator(TurnLightOff("turn_light_off"))

# 4. Register the termination check method
dp.add_termination_check(is_terminal)

# 5. Test execution flow.
print(f"Before decision process runs: {state.lamp=}, {state.switch=}")
dp.run_until_done()
print(f"After decision process runs: {state.lamp=}, {state.switch=}")

## Developer exercises
To understand the declarative nature of `cognition`, modify the variables in the block above to see what the engine does. 

### Exercise 1: What happens when the lamp is already off?

After the decision process is done running, the state of the lamp is `OFF`. Let's run the process again to see what happens when the lamp is. We need to reset `state.actuated` to `False` and re-initalize the decision process. `dp.reinit()` resets the execution engine.

In [ ]:
state.actuated = False
print(f"Before decision process runs: {state.lamp=}, {state.switch=}")
dp.reinit()
dp.run_until_done()
print(f"Before decision process runs: {state.lamp=}, {state.switch=}")

*Expected behavior*: The decision process raises a `DecisionProcessExecutionError` indicating that there are no potential `operators` that can be proposed. 

*Why does it happen?* This error is to be expected - we haven't told the decision process what to do when the light was already off. That is, the `can_perform` of `TurnLightOff` operator tests for light to be on.

*How can we fix this error?* We can redefine what `TurnLightOff` is by moving the test for if lamp is on to the `perform` method. The updated operator now defines MIDNIGHT mode behavior as *turn lamp off, if on or keep it off*

In [ ]:
# pylint: disable=function-redefined
class TurnLightOff(Operator[HomeState]):  # type: ignore[no-redef]
    """
    when: it is MIDNIGHT mode and the lamp is ON.
    then: disable the switch (and turn-off the lamp), and set the actuated flag.
    """

    def can_perform(self, state: HomeState, _io: IOContainer) -> bool:
        """propose when it is MIDNIGHT mode and the lamp is ON"""
        return state.mode is Mode.MIDNIGHT

    def perform(self, state: HomeState, _io: IOContainer) -> None:
        """apply to disable the switch (and turn-off the lamp), and set the actuated flag."""
        if state.lamp is Lamp.ON:
            state.switch = Switch.DISABLED
        state.actuated = True

Now, add this new definition of `TurnLightOff` to the decision process.

In [ ]:
dp.add_operator(TurnLightOff("turn_light_off"))

Let's run the decision process again. 

In [ ]:
state.actuated = False
print(f"Before decision process runs: {state.lamp=}, {state.switch=}")
dp.reinit()
dp.run_until_done()
print(f"Before decision process runs: {state.lamp=}, {state.switch=}")

The decision process now runs without errors. 

### Exercise 2: Write `EVENING` mode

The decision process we have built until now is incomplete. It defines behavior only for the `MIDNIGHT` mode.  Locate the instantiation of `State` class inside the block above and change the initial mode to `MODE.EVENING`. And, run the decision process again. 

In [ ]:
state.mode = Mode.EVENING
state.actuated = False
dp.reinit()
dp.run_until_done()

*Expected behavior*: You will see `DecisionProcessExecutionError`  from `cognition.core` with an `NO_PROPOSAL` indicator. 

*Why does this happen?* The engine evaluates `TurnLightOff.can_perform` and finds it returns `False` - which means that TurnLightOff operator should not be proposed. And, there is no other operator.

*How can we fix this?* We have to define another operator that encodes what the agent should do in the `EVENING`. Let's say, in the evening mode, the home assistant turns on the lamp if it is off and keeps it on if it is already on. You will need to add another operator as follows.

In [ ]:
class TurnLightOn(Operator[HomeState]):
    """keep light on."""

    def can_perform(self, state: HomeState, _io: IOContainer) -> bool:
        """propose when it's night mode and the lamp is physically on."""
        return state.mode is Mode.EVENING

    def perform(self, state: HomeState, _io: IOContainer) -> None:
        """apply to flip the actuator switch off (lamp off) and set the actuated flag"""
        if state.lamp is Lamp.OFF:
            state.switch = Switch.ENABLED
        state.actuated = True

In [ ]:
dp.add_operator(TurnLightOn("turn_light_on"))

In [ ]:
state.mode = Mode.EVENING
state.actuated = False
print(f"Before decision process runs: {state.lamp=}, {state.switch=} %")
dp.reinit()
dp.run_until_done()
print(f"Before decision process runs: {state.lamp=}, {state.switch=} %")

# Declarative vs imperative control of decisions

### 1. The imperative way - rigid control flow

In a traditional workflow engine, you write explicit, step-by-step instructions. The control flow is hardcoded directly into the sequence of execution.

A conceptual imperative version of our home assistant logic might look like this:

```python
# Conceptual Imperative Workflow
# A conceptual imperative look at handling multiple room modes
def control_smart_home_imperative(state):
    # Step 1: Evaluate the mode sequentially
    if state.mode is Mode.MIDNIGHT:
        # Step 2: Handle Midnight logic
        if state.lamp is Lamp.ON:
            state.switch = Switch.DISABLED
            state.lamp = Lamp.OFF
        state.actuated = True
        
    elif state.mode is Mode.EVENING:
        # Step 3: Handle Evening logic
        if state.lamp is Lamp.OFF:
            state.switch = Switch.ENABLED
            state.lamp = Lamp.ON
        state.actuated = True
```
While this looks simple with only two modes, it becomes highly brittle as a system grows. Notice the inherent structural issues:
- *Tight coupling*: The logic for `MIDNIGHT` and `EVENING` lives in the exact same conditional block. If a developer breaks a line of code inside the `MIDNIGHT` block, the entire control loop can crash, inadvertently breaking `EVENING` mode too. 
- *Hardcoded ordering*: The sequence is entirely fixed. If you need to evaluate safety or override rules (e.g., "If a security alert is triggered, ignore the current mode and turn all lights on"), you must manually restructure the entire nested if/elif/else tree.
- *The "Spaghetti" trap*: As you introduce more independent variables—such as an outdoor motion sensor, a vacation override, or checking whether it's a weekday—the number of intersecting logical paths multiplies exponentially. Your neat sequential script rapidly mutates into a dense, unreadable web of deeply nested conditionals. Tracing the lifecycle of a single flag requires scanning hundreds of lines of interleaved code, meaning a change to one feature almost always introduces subtle, hard-to-track bugs in another.

### 2. Declarative approach in `cognition` 

With `cognition`'s decision process, control flow is completely decoupled from agent's logic. Look back at how we implemented Exercise 2: we did not edit a single line of code inside `TurnLightOff`. We didn't open up an overarching orchestrator script to add an `elif` branch.Instead, we simply created an entirely isolated, self-contained rule called `TurnLightOn` and added it to the decision process.

Decision processes provide an elegant way to implement behavior. 
- *Continuous evaluation*: The cognition engine acts as a continuous control loop. It evaluates the state, identifies which operators are currently eligible, and executes them.
- *Immediate adaptation*: If an external factor changes the state (e.g., a human overrides the physical switch), the loop simply re-evaluates the environment at the next tick. It naturally triggers a different operator or retries the current one without needing explicit error-handling paths.
- *Frictionless scaling*: To add new behaviors (e.g., turning on outdoor security lights), you build and register a new `Operator`. Pre-existing execution chains will need minimal edits.

### Imperative v/s declarative at a glance

| Dimension | Imperative Workflows | Polycog Decision Processes |
| :--- | :--- | :--- |
| **Control Flow** | Hardcoded, sequential step-by-step paths. | Emergent; computed dynamically based on live state. |
| **Primary Focus** | **How** to execute (the exact chain of instructions). | **What** conditions must be met (triggers, effects, and goals). |
| **Environment Drift** | Fragile; requires explicit error paths and retry logic. | Resilient; naturally self-corrects on the next evaluation tick. |
| **Extensibility** | Refactoring requires modifying the parent orchestration block. | Extendable; add new operators with minimial re-touching of existing code. |
